# ONN Bench Backend — validation notebook

**Purpose:** validate every device control path and the core ONN contract —
a **temporal input matrix `X (T, h, w)`** goes to the **DMD**, propagates through
the optics (laser → DMD → mirror → SLM → camera), and comes back as the
**prediction matrix `Y (T, gh, gw)`** read at the detector.

All device logic lives in importable `.py` modules (`hardware/`, `onn/`) — this
notebook only drives them, so the future GUI reuses the exact same classes.

**Set `SIM = False` on the bench PC** (with pyserial / ALP4lib / slmsuite / PySpin
installed) to run against real hardware. With `SIM = True` everything runs on
simulator twins with a seeded fake optical transform.

In [1]:
import subprocess, os
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                      "--format=csv"], capture_output=True, text=True).stdout)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "(not set — all GPUs visible, code defaults to 0)"))

try:
    import torch
    print("torch sees:", torch.cuda.device_count(), "GPU(s); current:",
          torch.cuda.get_device_name(torch.cuda.current_device()) if torch.cuda.is_available() else "none")
except ImportError:
    pass

index, name, memory.used [MiB], memory.total [MiB], utilization.gpu [%]
0, NVIDIA GeForce RTX 4070 Ti, 918 MiB, 12282 MiB, 0 %

CUDA_VISIBLE_DEVICES = (not set — all GPUs visible, code defaults to 0)


In [2]:
SIM = False   # <-- flip to False on the bench PC

import time
import numpy as np
import matplotlib.pyplot as plt

from hardware.base import load_profile, SafetyLockError
from onn import patterns
from onn.forward import ONNForward, pool_to_grid, save_result

profile = load_profile("config/onn_nico.yaml")
print("profile loaded — laser default:", profile["laser"]["default_power_mw"], "mW")

profile loaded — laser default: 10.0 mW


## 1. Connect the bench

One factory call builds all four devices (simulated or real) from the profile.
Real drivers raise a clear ImportError if their vendor SDK is missing.

In [3]:
if SIM:
    from hardware.simulators import make_sim_bench
    bench, laser, dmd, slm, camera = make_sim_bench(profile, seed=0)
else:
    from hardware.laser import LaserOBIS
    from hardware.dmd import DmdViALUX
    from hardware.slm import SlmMeadowlark
    from hardware.camera import CameraFLIR
    laser = LaserOBIS(port=profile["laser"]["port"],
                      default_power_mw=profile["laser"]["default_power_mw"],
                      max_power_mw=profile["laser"]["max_power_mw"])
    dmd = DmdViALUX(macropixel=profile["dmd"]["macropixel"],
                    invert=profile["dmd"]["invert"],
                    picture_time_us=profile["dmd"]["picture_time_us"])
    slm = SlmMeadowlark(lut_file=profile["slm"]["lut_file"],
                        wfc_file=profile["slm"]["wfc_file"],
                        resolution=profile["slm"]["resolution"],
                        coverglass_threshold_v=profile["slm"]["coverglass"]["threshold_v"])
    camera = CameraFLIR(preset=profile["camera"])

for dev in (laser, dmd, slm, camera):
    dev.connect()
    print(f"{dev.name:>7}: {dev.status()}")

  laser: {'state': 'ready', 'wavelength_nm': 660.0, 'power_mw': 0.02, 'power_setpoint_mw': 10.0, 'diode_temp_c': 25.0, 'baseplate_temp_c': 20.1, 'faults': '00000000'}
Loading library: C:\Program Files\ALP-4.3\ALP-4.3 API/x64/alp4395.dll
DMD found, resolution = 1920 x 1080.
    dmd: {'state': 'ready', 'shape': (1080, 1920), 'sequence_loaded': False}
Validating DPI awareness...success
Constructing Blink SDK...success
Loading LUT file...    slm: {'state': 'ready', 'lut': 'slm5602_estimated_660nm_from_slm5838_635nm_HDMI.LUT', 'wfc': 'black.bmp', 'coverglass': 'managed by Blink controller (auto-adjust + hold)'}
        camera preset note — skipped: GammaEnable (SpinnakerException), AcqFrameRateEnable (SpinnakerException), AcquisitionFrameRate (SpinnakerException)
 camera: {'state': 'ready', 'acquiring': False, 'model': 'Grasshopper3 GS3-U3-89S6M', 'fps': 43.0107536315918}


In [4]:
import sys

print("connecting laser...", flush=True)
laser.connect();  print("laser OK", flush=True)

print("connecting dmd...", flush=True)
dmd.connect();    print("dmd OK", flush=True)

print("connecting slm...", flush=True)
slm.connect();    print("slm OK", flush=True)

print("connecting camera...", flush=True)
camera.connect(); print("camera OK", flush=True)

for dev in (laser, dmd, slm, camera):
    print(f"{dev.name:>7}: {dev.status()}")

connecting laser...
laser OK
connecting dmd...
Loading library: C:\Program Files\ALP-4.3\ALP-4.3 API/x64/alp4395.dll
DMD found, resolution = 1920 x 1080.
dmd OK
connecting slm...
Validating DPI awareness...success
Constructing Blink SDK...success
Loading LUT file...slm OK
connecting camera...
        camera preset note — skipped: GammaEnable (SpinnakerException), AcqFrameRateEnable (SpinnakerException), AcquisitionFrameRate (SpinnakerException)
camera OK
  laser: {'state': 'ready', 'wavelength_nm': 660.0, 'power_mw': 0.02, 'power_setpoint_mw': 10.0, 'diode_temp_c': 25.0, 'baseplate_temp_c': 20.1, 'faults': '00000000'}
    dmd: {'state': 'ready', 'shape': (1080, 1920), 'sequence_loaded': False}
    slm: {'state': 'ready', 'lut': 'slm5602_estimated_660nm_from_slm5838_635nm_HDMI.LUT', 'wfc': 'black.bmp', 'coverglass': 'managed by Blink controller (auto-adjust + hold)'}
 camera: {'state': 'ready', 'acquiring': False, 'model': 'Grasshopper3 GS3-U3-89S6M', 'fps': 43.0107536315918}


In [ ]:
if SIM:
    from hardware.simulators import make_sim_bench
    bench, laser, dmd, slm, camera = make_sim_bench(profile, seed=0)
else:
    from hardware.laser import LaserOBIS
    from hardware.dmd import DmdViALUX
    from hardware.slm import SlmMeadowlark
    from hardware.camera import CameraFLIR
    laser = LaserOBIS(port=profile["laser"]["port"],
                      default_power_mw=profile["laser"]["default_power_mw"],
                      max_power_mw=profile["laser"]["max_power_mw"])
    dmd = DmdViALUX(macropixel=profile["dmd"]["macropixel"],
                    invert=profile["dmd"]["invert"],
                    picture_time_us=profile["dmd"]["picture_time_us"])
    slm = SlmMeadowlark(lut_file=profile["slm"]["lut_file"],
                        wfc_file=profile["slm"]["wfc_file"],
                        resolution=profile["slm"]["resolution"],
                        coverglass_threshold_v=profile["slm"]["coverglass"]["threshold_v"])
    camera = CameraFLIR(preset=profile["camera"])

for dev in (laser, dmd, slm, camera):
    dev.connect()
    print(f"{dev.name:>7}: {dev.status()}")

In [ ]:
# ---- fix for OBIS unit-suffixed replies like '25.0C' ----
import re

def _f(s):
    m = re.search(r"-?\d+\.?\d*", str(s))
    return float(m.group()) if m else float("nan")

if SIM:
    from hardware.simulators import make_sim_bench
    bench, laser, dmd, slm, camera = make_sim_bench(profile, seed=0)
else:
    import hardware.laser as hl

    def _status(self):
        self._require_serial()
        return {
            "state": self.state.value,
            "wavelength_nm": _f(self._query("SYST:INF:WAV?")),
            "power_mw": _f(self._query("SOUR:POW:LEV?")) * 1000.0,
            "power_setpoint_mw": _f(self._query("SOUR:POW:LEV:IMM:AMPL?")) * 1000.0,
            "diode_temp_c": _f(self._query("SOUR:TEMP:DIOD?")),
            "baseplate_temp_c": _f(self._query("SOUR:TEMP:BAS?")),
            "faults": self._query("SYST:FAUL?"),
        }

    hl.LaserOBIS.status = _status
    hl.LaserOBIS.get_power_mw = lambda self: _f(self._query("SOUR:POW:LEV?")) * 1000.0

    from hardware.laser import LaserOBIS
    from hardware.dmd import DmdViALUX
    from hardware.slm import SlmMeadowlark
    from hardware.camera import CameraFLIR
    laser = LaserOBIS(port=profile["laser"]["port"],
                      default_power_mw=profile["laser"]["default_power_mw"],
                      max_power_mw=profile["laser"]["max_power_mw"])
    dmd = DmdViALUX(macropixel=profile["dmd"]["macropixel"],
                    invert=profile["dmd"]["invert"],
                    picture_time_us=profile["dmd"]["picture_time_us"])
    slm = SlmMeadowlark(lut_file=profile["slm"]["lut_file"],
                        wfc_file=profile["slm"]["wfc_file"],
                        resolution=profile["slm"]["resolution"],
                        coverglass_threshold_v=profile["slm"]["coverglass"]["threshold_v"])
    camera = CameraFLIR(preset=profile["camera"])

for dev in (laser, dmd, slm, camera):
    dev.connect()
    print(f"{dev.name:>7}: {dev.status()}")

## 2. Laser (OBIS 660-75FP)

Default operating power (10.0 mW) was already applied on connect. Verify
telemetry, the software power cap, and emission control.

In [ ]:
print("status:", laser.status())
assert abs(laser.get_power_mw() - profile["laser"]["default_power_mw"]) < 1e-6, "default power not applied!"

try:                       # the software cap must reject unsafe requests
    laser.set_power_mw(999)
except ValueError as e:
    print("power cap OK ->", e)

laser.emission_on()
print("emitting at", laser.get_power_mw(), "mW")

## 3. DMD (ViALUX V-Module)

Two things to prove:
1. **Named patterns** — e.g. `"white circle on black background"` generated in
   memory, no file browsing (the EasyProj replacement).
2. **`project_input(x)`** — a small binary input matrix upscaled to centered
   macropixels: the ONN input path.

In [ ]:
circle = patterns.dmd_pattern("white circle on black background", dmd.shape)
dmd.project(circle)

x_demo = (np.random.default_rng(3).random((4, 4)) > 0.5).astype(np.uint8)
frame_demo = dmd.project_input(x_demo)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
ax[0].imshow(circle, cmap="gray"); ax[0].set_title("named pattern: circle")
ax[1].imshow(x_demo, cmap="gray"); ax[1].set_title("input matrix x (4x4)")
ax[2].imshow(frame_demo, cmap="gray"); ax[2].set_title("x as DMD macropixels")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("dmd:", dmd.status())

## 4. SLM (Meadowlark Blink HDMI)

Validates: phase write (LUT + `.bmp` WFC applied inside the driver),
temperature readout, and the **coverglass voltage lockout** — it auto-adjusts
toward the threshold, latches, and then *any* attempt to change it raises
`SafetyLockError`. This lives in the driver, so no GUI code can bypass it.

In [ ]:
grating = patterns.slm_pattern("blazed grating",
                               (profile["slm"]["resolution"][1], profile["slm"]["resolution"][0]))
slm.write(grating)
print(f"SLM temp: {slm.get_temperature_c():.2f} C")

for i in range(4):                      # watch the auto-adjust latch the lock
    print(f"poll {i}: {slm.poll_coverglass()}")

try:
    slm.set_coverglass_v(6.0)           # must be refused once locked
except SafetyLockError as e:
    print("lockout OK ->", e)

## 5. Camera (FLIR Grasshopper3)

The whole preset (exposure, gain, gamma, black level, frame rate, mode) was
pushed in one call on connect — nobody hand-tunes SpinView nodes. Grab a frame
and check the detector sees the DMD pattern.

In [ ]:
camera.apply_preset(profile["camera"])   # idempotent re-apply
frame = camera.grab()
print("frame:", frame.shape, frame.dtype, "max =", frame.max())

plt.figure(figsize=(5, 3.5))
plt.imshow(frame, cmap="inferno"); plt.colorbar(label="counts")
plt.title("detector frame (current DMD input)"); plt.xticks([]); plt.yticks([])
plt.show()

## 6. The ONN forward pass — temporal X → prediction matrix Y

The core deliverable. Build a **time-varying input** `X (T, 4, 4)` — here a
moving bright pixel plus random binary noise, i.e. the input genuinely changes
with time — and run it through the optics:

for each t:  X[t] → DMD macropixels → settle → capture → pool over detector grid → Y[t]

`Y` has shape `(T, gh, gw)` — one prediction matrix per timestep.
*Timing note:* this loop is software-paced (fine for validation). For fast
temporal streams the upgrade is `dmd.project_sequence()` (onboard ALP timing)
plus camera hardware triggering — same interfaces.

In [ ]:
T, (h, w) = 12, profile["onn"]["input_shape"]
rng = np.random.default_rng(7)

X = (rng.random((T, h, w)) > 0.75).astype(np.uint8)   # temporal noise...
for t in range(T):                                     # ...plus a moving pixel
    X[t, t % h, (t * 2) % w] = 1

onn = ONNForward.from_profile(dmd, camera, profile)
result = onn.forward(X, keep_frames=True)
Y = result.Y
print("X", X.shape, "->", "Y", Y.shape)

In [ ]:
ncol = 6
fig, ax = plt.subplots(2, ncol, figsize=(2 * ncol, 4.4))
for i in range(ncol):
    t = i * (T // ncol)
    ax[0, i].imshow(X[t], cmap="gray", vmin=0, vmax=1); ax[0, i].set_title(f"X[t={t}]", fontsize=9)
    ax[1, i].imshow(Y[t], cmap="inferno", vmin=0, vmax=Y.max()); ax[1, i].set_title(f"Y[t={t}]", fontsize=9)
    for a in (ax[0, i], ax[1, i]): a.set_xticks([]); a.set_yticks([])
ax[0, 0].set_ylabel("input"); ax[1, 0].set_ylabel("prediction")
plt.suptitle("temporal input matrix -> prediction matrix at the detector")
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 3))
plt.plot(Y.reshape(T, -1))
plt.xlabel("timestep t"); plt.ylabel("Y element value")
plt.title("every detector-grid element over time"); plt.tight_layout(); plt.show()

In [ ]:
# sanity check: Y must actually depend on X (identical inputs -> similar Y,
# different inputs -> different Y beyond noise)
xa = X[0]
ya1, _, _ = onn.step(xa); ya2, _, _ = onn.step(xa)
yb, _, _ = onn.step(1 - xa)
same = np.abs(ya1 - ya2).mean(); diff = np.abs(ya1 - yb).mean()
print(f"repeatability |dY| = {same:.4f}   contrast |dY| = {diff:.4f}")
assert diff > 3 * same, "Y does not respond to X strongly enough vs noise"
print("forward pass OK: Y is a real function of X")

path = save_result(result, X, "data/run_sim.npz",
                   meta={"sim": SIM, "laser_mw": laser.get_power_mw(),
                         "grid": profile["onn"]["detector_grid"]})
print("saved ->", path)

## 7. Shutdown

Safe order: DMD freed, SLM blanked, laser emission off, everything disconnected.

In [ ]:
dmd.free(); slm.blank(); laser.emission_off()
for dev in (camera, slm, dmd, laser):
    dev.disconnect()
    print(f"{dev.name:>7}: {dev.status()['state']}")

---
### Bench-day checklist (SIM = False)
1. Install: `pip install pyserial ALP4lib slmsuite PySpin pillow pyyaml` (PySpin from FLIR/Teledyne installer; ALP + Blink DLLs from vendor installs).
2. Update `config/onn_nico.yaml`: real COM port, LUT path, coverglass threshold, and the optimal hyperparameters from `onn-nico-parameters`.
3. Confirm the SLM is configured as a second display before connecting.
4. Run top-to-bottom; every section is independent enough to debug one device at a time.